# 🇹🇳 Tunisia Tourism Intelligence — Exploratory Data Analysis
**DS2 Project | Business / Finance**  
**Sources:** World Bank, INS, ONTT  
**Period:** 2000–2024

---
### Notebook Structure
1. Setup & Data Loading
2. Data Quality Check
3. Annual Overview Analysis
4. Monthly Seasonality Analysis
5. Source Markets Analysis
6. Regional Analysis
7. KPI Computation & Export (clean CSVs for Streamlit)

## 1. Setup & Data Loading

In [ ]:
# Imports 
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.max_columns', 20)

print('✅ Libraries loaded successfully')

✅ Libraries loaded successfully


In [ ]:
# Load all datasets 
# Ajustement du chemin
DATA_PATH = '../data/raw/'

annual   = pd.read_csv(DATA_PATH + 'annual_overview_final.csv')
monthly  = pd.read_csv(DATA_PATH + 'monthly_arrivals.csv')
markets  = pd.read_csv(DATA_PATH + 'source_markets.csv')
regional = pd.read_csv(DATA_PATH + 'regional_data.csv')

print(f'Annual data   : {annual.shape[0]} rows × {annual.shape[1]} cols  ({annual.year.min()}–{annual.year.max()})')
print(f'Monthly data  : {monthly.shape[0]} rows × {monthly.shape[1]} cols')
print(f'Source markets: {markets.shape[0]} rows × {markets.shape[1]} cols')
print(f'Regional data : {regional.shape[0]} rows × {regional.shape[1]} cols')

Annual data   : 25 rows × 8 cols  (2000–2024)
Monthly data  : 119 rows × 4 cols
Source markets: 33 rows × 4 cols
Regional data : 11 rows × 8 cols


## 2. Data Quality Check

In [ ]:
# Missing values check 
print('=== ANNUAL — Missing values ===')
print(annual.isnull().sum())
print()
print('=== MONTHLY — Missing values ===')
print(monthly.isnull().sum())
print()
print('=== MARKETS — Missing values ===')
print(markets.isnull().sum())

=== ANNUAL — Missing values ===
year                        0
arrivals_thousands          0
receipts_musd               0
receipts_mtnd               0
tourism_pct_exports         0
revenue_per_visitor_usd     0
source                      0
investment_mtnd            16
dtype: int64

=== MONTHLY — Missing values ===
year                  0
month                 0
month_name            0
arrivals_thousands    0
dtype: int64

=== MARKETS — Missing values ===
year                  0
country               0
arrivals_thousands    0
region                0
dtype: int64


In [ ]:
# Annual data
print('Annual dataset overview:')
annual.info()
print()
annual.head()

Annual dataset overview:
<class 'pandas.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   year                     25 non-null     int64  
 1   arrivals_thousands       25 non-null     int64  
 2   receipts_musd            25 non-null     float64
 3   receipts_mtnd            25 non-null     int64  
 4   tourism_pct_exports      25 non-null     float64
 5   revenue_per_visitor_usd  25 non-null     float64
 6   source                   25 non-null     str    
 7   investment_mtnd          9 non-null      float64
dtypes: float64(4), int64(3), str(1)
memory usage: 1.9 KB



,year,arrivals_thousands,receipts_musd,receipts_mtnd,tourism_pct_exports,revenue_per_visitor_usd,source,investment_mtnd
0,2000,5058,"1,977.00",2372,23.10,390.90,World Bank,NaN
1,2001,5387,"2,061.00",2679,21.78,382.60,World Bank,NaN
2,2002,5064,"1,831.00",2600,19.34,361.60,World Bank,NaN
3,2003,5114,"1,935.00",2883,17.77,378.40,World Bank,NaN
4,2004,5998,"2,432.00",3624,18.01,405.50,World Bank,NaN


In [ ]:
# Ajout des colonnes

annual = annual.sort_values('year').reset_index(drop=True)
annual['arrivals_yoy_pct'] = annual['arrivals_thousands'].pct_change() * 100

base_2019 = annual.loc[annual['year'] == 2019, 'arrivals_thousands'].values[0]
annual['recovery_index'] = (annual['arrivals_thousands'] / base_2019 * 100).round(1)

# Labels des crises 
crisis_years = {
    2011: 'Revolution',
    2015: 'Terror Attacks',
    2020: 'COVID-19',
    2023: 'Record Recovery'
}
annual['event'] = annual['year'].map(crisis_years)

print('Derived columns added:')
print(annual[['year','arrivals_thousands','arrivals_yoy_pct','recovery_index','event']].tail(10).to_string(index=False))

Derived columns added:
 year  arrivals_thousands  arrivals_yoy_pct  recovery_index           event
 2015                5359            -25.18           56.80  Terror Attacks
 2016                5724              6.81           60.70             NaN
 2017                7052             23.20           74.80             NaN
 2018                8299             17.68           88.00             NaN
 2019                9429             13.62          100.00             NaN
 2020                2012            -78.66           21.30        COVID-19
 2021                2187              8.70           23.20             NaN
 2022                6067            177.41           64.30             NaN
 2023                9370             54.44           99.40 Record Recovery
 2024                9800              4.59          103.90             NaN


## 3. Annual Overview Analysis

In [ ]:
# Sommaire des KPI
latest = annual[annual['year'] == annual['year'].max()].iloc[0]
prev   = annual[annual['year'] == annual['year'].max() - 1].iloc[0]
peak   = annual.loc[annual['arrivals_thousands'].idxmax()]

print(f"""╔══════════════════════════════════════════════════════╗
║         TUNISIA TOURISM — KEY METRICS ({int(latest.year)})         ║
╠══════════════════════════════════════════════════════╣
║ Tourist Arrivals    : {latest.arrivals_thousands:>8,.0f}K visitors     ║
║ Tourism Revenue     : ${latest.receipts_musd:>8,.0f}M USD          ║
║ Revenue per Visitor : ${latest.revenue_per_visitor_usd:>8.0f} USD            ║
║ Tourism % of Exports: {latest.tourism_pct_exports:>8.1f}%               ║
║ Recovery vs 2019    : {latest.recovery_index:>8.1f}% of peak          ║
║ All-time Peak Year  : {int(peak.year):>8} ({peak.arrivals_thousands:,.0f}K)     ║
╚══════════════════════════════════════════════════════╝""")

╔══════════════════════════════════════════════════════╗
║         TUNISIA TOURISM — KEY METRICS (2024)         ║
╠══════════════════════════════════════════════════════╣
║ Tourist Arrivals    :    9,800K visitors     ║
║ Tourism Revenue     : $   2,650M USD          ║
║ Revenue per Visitor : $     270 USD            ║
║ Tourism % of Exports:     14.0%               ║
║ Recovery vs 2019    :    103.9% of peak          ║
║ All-time Peak Year  :     2024 (9,800K)     ║
╚══════════════════════════════════════════════════════╝


In [ ]:
# ── Chart 1: Arrivals over time with crisis annotations 
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=annual['year'], y=annual['arrivals_thousands'],
    mode='lines+markers',
    line=dict(color='#E74C3C', width=3),
    marker=dict(size=7, color='#E74C3C'),
    name='Arrivals (thousands)',
    hovertemplate='<b>%{x}</b><br>Arrivals: %{y:,.0f}K<extra></extra>'
))

# Annotations des crises 
events = {
    2011: ('Revolution', '#E67E22'),
    2015: ('Bardo & Sousse\nAttacks', '#8E44AD'),
    2020: ('COVID-19', '#2C3E50'),
    2023: ('Record\nRecovery 🏆', '#27AE60')
}
for yr, (label, color) in events.items():
    val = annual.loc[annual['year']==yr, 'arrivals_thousands'].values
    if len(val):
        fig.add_annotation(x=yr, y=val[0], text=label,
            showarrow=True, arrowhead=2, arrowcolor=color,
            font=dict(size=10, color=color), ax=0, ay=-50)

fig.add_hrect(y0=0, y1=3000, fillcolor='red', opacity=0.05, line_width=0)

fig.update_layout(
    title='Tunisia Tourist Arrivals 2000–2024 — A Story of Resilience',
    xaxis_title='Year', yaxis_title='Arrivals (thousands)',
    template='plotly_white', height=450,
    hovermode='x unified'
)
fig.show()

In [ ]:
# ── Chart 2: Revenue USD vs Arrivals (dual axis) 
fig = make_subplots(specs=[[{'secondary_y': True}]])

fig.add_trace(go.Bar(
    x=annual['year'], y=annual['arrivals_thousands'],
    name='Arrivals (K)', marker_color='#3498DB', opacity=0.7
), secondary_y=False)

fig.add_trace(go.Scatter(
    x=annual['year'], y=annual['receipts_musd'],
    name='Revenue (M USD)', line=dict(color='#E74C3C', width=3),
    mode='lines+markers'
), secondary_y=True)

fig.update_layout(
    title='Arrivals vs Revenue — Volume vs Value Gap',
    template='plotly_white', height=420,
    legend=dict(x=0.01, y=0.99)
)
fig.update_yaxes(title_text='Arrivals (thousands)', secondary_y=False)
fig.update_yaxes(title_text='Revenue (M USD)', secondary_y=True)
fig.show()

In [ ]:
# ── Chart 3: Revenue per visitor (quality of tourism) 
fig = px.bar(
    annual[annual['year'] >= 2000],
    x='year', y='revenue_per_visitor_usd',
    color='revenue_per_visitor_usd',
    color_continuous_scale='RdYlGn',
    title='Revenue per Visitor (USD) — Are We Attracting High-Value Tourists?',
    labels={'revenue_per_visitor_usd': 'USD / Visitor', 'year': 'Year'},
    text='revenue_per_visitor_usd'
)
fig.update_traces(texttemplate='$%{text:.0f}', textposition='outside')
fig.update_layout(template='plotly_white', height=420, coloraxis_showscale=False)
fig.show()

print()
print('💡 Insight: Revenue per visitor peaked early (2008: $554) then declined.')
print('   More tourists since 2017 but lower spending per head — mass vs quality tourism debate.')


💡 Insight: Revenue per visitor peaked early (2008: $554) then declined.
   More tourists since 2017 but lower spending per head — mass vs quality tourism debate.


In [ ]:
# ── Chart 4: Recovery Index vs 2019 
post_covid = annual[annual['year'] >= 2019].copy()

fig = go.Figure()
fig.add_hline(y=100, line_dash='dash', line_color='gray', annotation_text='2019 Baseline (100%)')
fig.add_trace(go.Bar(
    x=post_covid['year'], y=post_covid['recovery_index'],
    marker_color=['#27AE60' if v >= 100 else '#E74C3C' if v < 50 else '#E67E22'
                  for v in post_covid['recovery_index']],
    text=[f'{v:.1f}%' for v in post_covid['recovery_index']],
    textposition='outside',
    name='Recovery Index'
))
fig.update_layout(
    title='Post-COVID Recovery Index vs 2019 Baseline',
    xaxis_title='Year', yaxis_title='Recovery Index (2019 = 100)',
    template='plotly_white', height=380
)
fig.show()

## 4. Monthly Seasonality Analysis

In [ ]:
# ── Chart 5: Heatmap — month × year arrivals 
heatmap_data = monthly.pivot(index='month_name', columns='year', values='arrivals_thousands')

# Order months 
month_order = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']
heatmap_data = heatmap_data.reindex(month_order)

fig = px.imshow(
    heatmap_data,
    color_continuous_scale='YlOrRd',
    title='Monthly Arrivals Heatmap — Seasonality Pattern (2015–2024)',
    labels=dict(x='Year', y='Month', color='Arrivals (K)'),
    aspect='auto'
)
fig.update_layout(template='plotly_white', height=420)
fig.show()

print()
print('💡 Insight: July & August consistently dominate — summer peak accounts for ~35% of annual arrivals.')
print('   COVID collapse visible in 2020: April–June essentially zero.')


💡 Insight: July & August consistently dominate — summer peak accounts for ~35% of annual arrivals.
   COVID collapse visible in 2020: April–June essentially zero.


In [ ]:
# ── Chart 6: Average monthly profile (seasonality curve) 
avg_monthly = monthly.groupby(['month', 'month_name'])['arrivals_thousands'].mean().reset_index()
avg_monthly = avg_monthly.sort_values('month')

fig = px.area(
    avg_monthly, x='month_name', y='arrivals_thousands',
    title='Average Monthly Arrivals Profile — The Seasonality Problem',
    labels={'arrivals_thousands': 'Avg Arrivals (K)', 'month_name': ''},
    color_discrete_sequence=['#E74C3C']
)
fig.update_layout(template='plotly_white', height=380)
fig.show()

# Concentration par saison
peak_months = monthly[monthly['month'].isin([7,8])]['arrivals_thousands'].sum()
total = monthly['arrivals_thousands'].sum()
print(f'\n💡 July + August = {peak_months/total*100:.1f}% of all arrivals in dataset')
print('   Off-peak (Nov–Feb) = structurally underutilized — key business recommendation')


💡 July + August = 35.4% of all arrivals in dataset
   Off-peak (Nov–Feb) = structurally underutilized — key business recommendation


In [ ]:
# ── Chart 7: Year comparison lines (2019 vs 2022 vs 2023) 
compare_years = monthly[monthly['year'].isin([2019, 2022, 2023])].copy()
compare_years = compare_years.sort_values(['year','month'])

fig = px.line(
    compare_years, x='month_name', y='arrivals_thousands',
    color='year', color_discrete_sequence=['#27AE60','#E67E22','#E74C3C'],
    title='Monthly Recovery: 2022 & 2023 vs 2019 Pre-COVID Peak',
    labels={'arrivals_thousands': 'Arrivals (K)', 'month_name': '', 'year': 'Year'},
    markers=True
)
fig.update_layout(template='plotly_white', height=400)
fig.show()

## 5. Source Markets Analysis

In [ ]:
# ── Chart 8: Top source markets 2023 
m2023 = markets[markets['year'] == 2023].sort_values('arrivals_thousands', ascending=True)

colors = ['#E74C3C' if c in ['Algeria','Libya','Morocco'] else '#3498DB'
          for c in m2023['country']]

fig = go.Figure(go.Bar(
    x=m2023['arrivals_thousands'], y=m2023['country'],
    orientation='h',
    marker_color=colors,
    text=m2023['arrivals_thousands'].apply(lambda x: f'{x:,.0f}K'),
    textposition='outside'
))
fig.update_layout(
    title='Top Source Markets 2023 — 🔴 Africa | 🔵 Europe',
    xaxis_title='Arrivals (thousands)', template='plotly_white', height=420
)
fig.show()

# Concentration risk
top2 = m2023.nlargest(2, 'arrivals_thousands')['arrivals_thousands'].sum()
total_2023 = m2023['arrivals_thousands'].sum()
print(f'\n💡 Algeria + Libya = {top2/total_2023*100:.1f}% of all arrivals in 2023')
print('   High concentration risk — geopolitical instability in these countries = sector shock')


💡 Algeria + Libya = 54.1% of all arrivals in 2023
   High concentration risk — geopolitical instability in these countries = sector shock


In [ ]:
# ── Chart 9: Market share evolution 2019 → 2023 
fig = px.bar(
    markets[markets['year'].isin([2019, 2022, 2023])],
    x='year', y='arrivals_thousands', color='country',
    title='Source Market Evolution — Which Countries Are Coming Back?',
    labels={'arrivals_thousands': 'Arrivals (K)', 'year': 'Year'},
    barmode='group'
)
fig.update_layout(template='plotly_white', height=420)
fig.show()

In [ ]:
# ── Chart 10: Region breakdown pie 2023 
region_2023 = markets[markets['year']==2023].groupby('region')['arrivals_thousands'].sum().reset_index()

fig = px.pie(
    region_2023, values='arrivals_thousands', names='region',
    title='Tourism by Region of Origin — 2023',
    color_discrete_sequence=['#E74C3C','#3498DB','#95A5A6'],
    hole=0.4
)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.update_layout(template='plotly_white', height=380)
fig.show()

## 6. Regional Analysis

In [ ]:
# ── Chart 11: Hotel capacity by governorate 
regional_sorted = regional.sort_values('hotel_beds', ascending=True)

fig = go.Figure()
fig.add_trace(go.Bar(
    y=regional_sorted['governorate'],
    x=regional_sorted['hotel_capacity_pct_2019'],
    name='2019 (Pre-COVID)', orientation='h',
    marker_color='#BDC3C7'
))
fig.add_trace(go.Bar(
    y=regional_sorted['governorate'],
    x=regional_sorted['hotel_capacity_pct_2023'],
    name='2023', orientation='h',
    marker_color='#E74C3C'
))
fig.update_layout(
    barmode='group',
    title='Hotel Occupancy Rate by Governorate — 2019 vs 2023',
    xaxis_title='Occupancy Rate (%)',
    template='plotly_white', height=420
)
fig.show()

print('\n💡 Djerba leads recovery (88% in 2023 vs 80% in 2019) — island tourism boom post-COVID')
print('   Northern regions (Tabarka, Bizerte) remain underutilized — opportunity gap')


💡 Djerba leads recovery (88% in 2023 vs 80% in 2019) — island tourism boom post-COVID
   Northern regions (Tabarka, Bizerte) remain underutilized — opportunity gap


In [ ]:
# ── Chart 12: Map of governorates (hotel beds bubble) 
fig = px.scatter_mapbox(
    regional,
    lat='lat', lon='lon',
    size='hotel_beds',
    color='hotel_capacity_pct_2023',
    hover_name='governorate',
    hover_data={'hotel_beds': True, 'hotel_capacity_pct_2023': True, 'tourism_type': True},
    color_continuous_scale='RdYlGn',
    size_max=40,
    zoom=5.5,
    center={'lat': 33.8, 'lon': 9.5},
    mapbox_style='carto-positron',
    title='Tunisia Tourism Map — Hotel Capacity & Occupancy 2023'
)
fig.update_layout(height=520, template='plotly_white')
fig.show()

## 7. KPI Computation & Export

In [ ]:
# ── Compute les KPIs and save clean CSVs for the Streamlit app 
import os
CLEAN_PATH = '../data/clean/'
os.makedirs(CLEAN_PATH, exist_ok=True)

# 1. Annual clean
annual.to_csv(CLEAN_PATH + 'annual_clean.csv', index=False)

# 2. Monthly with season label
def get_season(m):
    if m in [12, 1, 2]: return 'Winter'
    elif m in [3, 4, 5]: return 'Spring'
    elif m in [6, 7, 8]: return 'Summer'
    else: return 'Autumn'

monthly['season'] = monthly['month'].apply(get_season)
monthly.to_csv(CLEAN_PATH + 'monthly_clean.csv', index=False)

# 3. Markets clean
markets.to_csv(CLEAN_PATH + 'markets_clean.csv', index=False)

# 4. Regional clean
regional.to_csv(CLEAN_PATH + 'regional_clean.csv', index=False)

# 5. Seasonality summary
seasonality = monthly.groupby(['season','month']).agg(
    avg_arrivals=('arrivals_thousands','mean'),
    total_arrivals=('arrivals_thousands','sum')
).reset_index().sort_values('month')
seasonality.to_csv(CLEAN_PATH + 'seasonality_clean.csv', index=False)

# 6. KPI scorecard
latest_yr = annual['year'].max()
row = annual[annual['year'] == latest_yr].iloc[0]
prev_row = annual[annual['year'] == latest_yr - 1].iloc[0]

scorecard = pd.DataFrame({
    'kpi': ['Arrivals (K)', 'Revenue (M USD)', 'Revenue/Visitor (USD)',
             'Tourism % Exports', 'Recovery Index', 'Investment (M TND)'],
    'value_latest': [
        row['arrivals_thousands'], row['receipts_musd'],
        row['revenue_per_visitor_usd'], row['tourism_pct_exports'],
        row['recovery_index'], row['investment_mtnd']
    ],
    'value_prev': [
        prev_row['arrivals_thousands'], prev_row['receipts_musd'],
        prev_row['revenue_per_visitor_usd'], prev_row['tourism_pct_exports'],
        prev_row['recovery_index'], prev_row['investment_mtnd']
    ],
    'year': latest_yr
})
scorecard['delta_pct'] = ((scorecard['value_latest'] - scorecard['value_prev'])
                           / scorecard['value_prev'] * 100).round(1)
scorecard.to_csv(CLEAN_PATH + 'kpi_scorecard.csv', index=False)

print('✅ All clean datasets exported to data/clean/')
print()
for f in os.listdir(CLEAN_PATH):
    print(f'   📄 {f}')
print()
print('=== KPI Scorecard ===')
print(scorecard.to_string(index=False))

✅ All clean datasets exported to data/clean/

   📄 annual_clean.csv
   📄 kpi_scorecard.csv
   📄 markets_clean.csv
   📄 monthly_clean.csv
   📄 regional_clean.csv
   📄 seasonality_clean.csv

=== KPI Scorecard ===
                  kpi  value_latest  value_prev  year  delta_pct
         Arrivals (K)      9,800.00    9,370.00  2024       4.60
      Revenue (M USD)      2,650.00    2,436.00  2024       8.80
Revenue/Visitor (USD)        270.00      260.00  2024       3.80
    Tourism % Exports         14.00       13.50  2024       3.70
       Recovery Index        103.90       99.40  2024       4.50
   Investment (M TND)        205.00      150.00  2024      36.70


## Summary of Key Findings

| # | Finding | Business Implication |
|---|---------|---------------------|
| 1 | Tunisia reached 9.8M arrivals in 2024 — surpassing 2019 peak | Full recovery achieved, sector back to pre-COVID strength |
| 2 | Revenue per visitor declined from $554 (2008) to ~$270 (2024) | More tourists, lower spend — shift toward mass tourism |
| 3 | Algeria + Libya = ~47% of all arrivals | High geopolitical concentration risk |
| 4 | July–August = ~35% of annual arrivals | Extreme seasonality — winter capacity wasted |
| 5 | Djerba leads hotel recovery at 88% occupancy | Island tourism resilience model |
| 6 | Tourism investments collapsed post-attacks and COVID | Infrastructure gap risks limiting future growth |
